# Lab 02A: Agentic Workflow Patterns (Ollama Implementation)

This lab implements the five agentic workflow patterns described in Anthropic's engineering blog post [*Building Effective Agents*](https://www.anthropic.com/engineering/building-effective-agents).

All examples run **locally via Ollama** (zero API cost).

---

### Patterns covered

| # | Pattern | Core idea |
|---|---------|----------|
| 1 | **Prompt Chaining** | Sequential steps where each LLM call feeds the next |
| 2 | **Routing** | Classify input → dispatch to a specialised prompt/model |
| 3 | **Parallelization** | Run multiple LLM calls simultaneously, then aggregate |
| 4 | **Orchestrator-Workers** | Central LLM breaks down a task and delegates to workers |
| 5 | **Evaluator-Optimizer** | Generate → Evaluate → Refine loop |

> **Key insight from Anthropic:** Start simple. Only add complexity when it demonstrably improves outcomes.


## 0) Setup

Same environment as Lab 01. Make sure Ollama is running and your model is available.


In [1]:
import os
import asyncio
import json
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL    = os.getenv("OLLAMA_MODEL", "qwen3:8b")

client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL.rstrip('/')}/v1",
    api_key=os.getenv("OLLAMA_API_KEY", "ollama"),
)

def chat(messages: list[dict], temperature: float = 0.7) -> str:
    """Thin wrapper: returns the assistant reply as a plain string."""
    response = client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

print("Model:", OLLAMA_MODEL)
print("Ready ✓")


Model: qwen3:8b
Ready ✓


---
## Pattern 1 — Prompt Chaining

**Concept:** Decompose a complex task into a fixed sequence of smaller LLM calls. Each step's output becomes the next step's input. A *gate* (programmatic check) can be inserted between steps to catch problems early.

```
User Input → [Step 1 LLM] → output_1 → (gate?) → [Step 2 LLM] → output_2 → …
```

**Use when:** the task can be cleanly decomposed into sequential subtasks, and you want to trade latency for higher accuracy per step.

**Example below:** Blog post pipeline — outline → gate check → full draft → polish.


In [2]:
TOPIC = "Why vector databases matter for RAG applications"

# ── Step 1: Generate an outline ──────────────────────────────────────────────
outline = chat([
    {"role": "system", "content": "You are a technical content strategist."},
    {"role": "user",   "content": f"Write a 3-section outline for a blog post about: {TOPIC}. Return only the outline."},
])
display(Markdown("### Step 1 — Outline\n\n" + outline))

# ── Gate: check the outline has exactly 3 sections ───────────────────────────
section_count = outline.count("\n#") + outline.count("\n1.") + outline.count("\n2.")
gate_pass = len(outline.strip()) > 50   # simple length guard
print(f"\n🔍 Gate check — outline non-empty: {gate_pass}")
assert gate_pass, "Outline too short — halting pipeline."

# ── Step 2: Write the draft ───────────────────────────────────────────────────
draft = chat([
    {"role": "system",    "content": "You are a senior technical writer."},
    {"role": "user",      "content": f"Topic: {TOPIC}"},
    {"role": "assistant", "content": outline},
    {"role": "user",      "content": "Expand the outline into a concise 200-word blog post draft."},
])
display(Markdown("### Step 2 — Draft\n\n" + draft))

# ── Step 3: Polish for a non-technical audience ───────────────────────────────
polished = chat([
    {"role": "system", "content": "You are an editor who simplifies technical writing."},
    {"role": "user",   "content": f"Rewrite the following blog post so it is engaging for a non-technical audience. Keep it under 150 words.\n\n{draft}"},
])
display(Markdown("### Step 3 — Polished\n\n" + polished))


### Step 1 — Outline

1. Introduction to RAG Applications and the Role of Vector Databases  
   - Overview of Retrieval-Augmented Generation (RAG) and its reliance on semantic search  
   - Introduction to vector databases as a key infrastructure component  

2. Why Vector Databases Matter for RAG: Core Advantages  
   - Scalability and efficiency in handling high-dimensional embeddings  
   - Real-time retrieval performance for dynamic query demands  
   - Enhanced accuracy through semantic similarity and proximity-based searches  

3. Challenges and Future Trends in Vector Database Integration with RAG  
   - Balancing scalability with computational costs and storage demands  
   - Emerging trends: hybrid models, AI-driven optimization, and edge computing  
   - The evolving role of vector databases in next-generation RAG systems


🔍 Gate check — outline non-empty: True


### Step 2 — Draft

**Why Vector Databases Matter for RAG Applications**  

Retrieval-Augmented Generation (RAG) systems combine semantic search with generative models to deliver accurate, context-aware responses. At their core, RAG applications rely on vector databases to transform raw text into high-dimensional embeddings, enabling efficient semantic similarity searches. These databases are critical for scaling RAG systems, as they handle vast datasets and dynamic queries with low latency.  

Vector databases excel in managing high-dimensional embeddings, ensuring scalability without compromising performance. Their ability to perform real-time, proximity-based searches enhances accuracy, allowing RAG to retrieve the most relevant information swiftly. This is vital for applications like customer service, research, and content generation, where speed and precision are paramount.  

However, challenges remain, such as balancing scalability with computational costs and storage demands. Emerging trends like hybrid models, AI-driven optimization, and edge computing are reshaping vector database integration, paving the way for smarter, more efficient RAG systems. As RAG evolves, vector databases will remain a cornerstone, bridging the gap between data and intelligent, real-world applications.  

*(Word count: 200)*

### Step 3 — Polished

**Why Vector Databases Power Smart AI Systems**  

Imagine a chatbot that knows exactly what to say—thanks to a mix of search and AI. Retrieval-Augmented Generation (RAG) systems work like this: they use vector databases to turn text into "digital fingerprints," letting AI quickly find the most relevant info. These databases are the backbone of RAG, handling massive data and real-time queries efficiently.  

They excel at spotting patterns in text, enabling fast, accurate answers for apps like customer service or research. Yet, challenges like cost and storage remain. New trends, like AI-driven tools and edge computing, are making these systems smarter and more scalable. As AI evolves, vector databases will keep bridging the gap between data and real-world insights—making technology feel more human than ever.  

*(Word count: 149)*

---
## Pattern 2 — Routing

**Concept:** A *classifier* LLM (or rule) reads the input and assigns it a category. A second LLM call — with a prompt specialised for that category — handles the actual response.

```
User Input → [Classifier LLM] → category → [Specialist LLM for category] → Response
```

**Use when:** you have distinct input types that benefit from different prompts, tools, or even different models.

**Example below:** Customer support triage — billing, technical, or general queries each get a tailored response.


In [3]:
SPECIALIST_PROMPTS = {
    "billing":   "You are a billing support specialist. Be empathetic and solution-focused.",
    "technical": "You are a senior technical support engineer. Give precise, step-by-step answers.",
    "general":   "You are a friendly customer success agent. Be warm and concise.",
}

def classify_query(user_query: str) -> str:
    """Return one of: billing | technical | general"""
    result = chat(
        [
            {"role": "system", "content": (
                "Classify the following customer query into exactly one category: "
                "billing, technical, or general. "
                "Reply with the single word only."
            )},
            {"role": "user", "content": user_query},
        ],
        temperature=0,
    ).strip().lower()
    # Normalise — model might add punctuation
    for key in SPECIALIST_PROMPTS:
        if key in result:
            return key
    return "general"

def routed_response(user_query: str) -> str:
    category = classify_query(user_query)
    system_prompt = SPECIALIST_PROMPTS[category]
    answer = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query},
    ])
    return category, answer

# ── Test three query types ────────────────────────────────────────────────────
queries = [
    "I was charged twice on my last invoice.",
    "My API requests keep returning a 429 error even though I'm below the limit.",
    "What are your business hours?",
]

for q in queries:
    category, answer = routed_response(q)
    display(Markdown(f"**Query:** {q}\n\n**Routed to:** `{category}`\n\n**Response:** {answer}\n\n---"))


**Query:** I was charged twice on my last invoice.

**Routed to:** `billing`

**Response:** I'm sorry to hear you experienced duplicate charges—it’s frustrating, and I appreciate you bringing this to our attention. Let’s resolve this together!  

Could you share the following details so I can investigate and correct this?  
1. **Invoice numbers** (if available) for both charges.  
2. **Dates** of the charges.  
3. **Amounts** charged each time.  
4. Any **transaction IDs** or payment method details (e.g., card ending in XXXX).  

Once I have this information, I’ll:  
- Check your account for duplicate entries.  
- Initiate a refund for the overcharged amount.  
- Provide steps to prevent this from happening again.  

Thank you for your patience—this is not your fault, and I’ll ensure this is resolved quickly. Let me know how I can assist further! 😊

---

**Query:** My API requests keep returning a 429 error even though I'm below the limit.

**Routed to:** `technical`

**Response:** A **429 Too Many Requests** error typically indicates your API requests are exceeding the rate limit, even if you believe you're within the allowed threshold. Here's a step-by-step guide to diagnose and resolve the issue:

---

### **1. Verify the Rate Limit Details**
   - **Check the API's documentation** for exact limit values (e.g., 100 requests/hour, 50 requests/minute) and the **time window** (e.g., 1 hour, 1 minute).
   - Look for **rate limit headers** in the API response (e.g., `X-RateLimit-Remaining`, `X-RateLimit-Reset`). These often provide real-time usage metrics.

---

### **2. Analyze Request Frequency**
   - **Calculate your request rate**:  
     Use tools like `curl`, Postman, or a script to log timestamps of each request. Count how many requests you’re making per second/minute/hour.
   - **Compare with the limit**:  
     If your rate exceeds the API's stated limit (even slightly), adjust your request frequency. For example:
     - If the limit is **100 requests/hour**, avoid making 100+ requests in a 5-minute window.
     - Use a **rate-limiting library** (e.g., `axios-rate-limit`, `express-rate-limit`) to enforce compliance.

---

### **3. Check for Duplicate or Misconfigured Keys**
   - **Same API key used across services**: If multiple services (e.g., web app, mobile app, server) use the same key, their combined usage may exceed the limit.
   - **Ensure unique keys**: Assign separate API keys to different services or users to isolate usage.

---

### **4. Investigate IP Address or User-Agent Rate Limiting**
   - **IP-based limits**: Some APIs rate-limit based on the client's IP address. If your IP is shared (e.g., a shared hosting provider), check if others are also hitting the limit.
   - **User-Agent headers**: Some APIs restrict requests based on the `User-Agent` string. Use a consistent, identifiable `User-Agent` to avoid being flagged.

---

### **5. Examine Third-Party Integrations**
   - **Webhooks or services**: If you’re using a third-party service (e.g., Zapier, Cloudflare) to make requests, it might be hitting the rate limit unintentionally.
   - **Check their usage policies**: Ensure the third-party service respects the API’s rate limits.

---

### **6. Handle Token Authentication Correctly**
   - **OAuth tokens**: If you’re using OAuth, ensure the token is valid and not expired. Invalid tokens may trigger rate limits during authentication.
   - **Token rotation**: Some APIs rate-limit based on the token’s source. Rotate tokens periodically if required.

---

### **7. Use Retry-After Headers**
   - If the API returns a `Retry-After` header, wait the specified time before resuming requests (e.g., `Retry-After: 60` means wait 60 seconds).

---

### **8. Contact the API Provider**
   - If the issue persists, reach out to the API’s support team with:
     - Your **API key or credentials** (if allowed).
     - **Request logs** showing timestamps and endpoints.
     - **Rate limit headers** from the API response.
   - They may identify unexpected usage patterns or temporary rate limit adjustments.

---

### **Example Debugging Script (Python)**
```python
import requests
import time

headers = {
    "Authorization": "Bearer YOUR_API_TOKEN",
    "User-Agent": "MyApp/1.0"
}

for i in range(100):
    response = requests.get("https://api.example.com/data", headers=headers)
    print(f"Request {i+1}: {response.status_code}")
    time.sleep(1)  # Ensure requests are spaced out
```

---

By systematically checking these areas, you should identify the root cause of the 429 error and adjust your usage accordingly.

---

**Query:** What are your business hours?

**Routed to:** `general`

**Response:** We're open from **9:00 AM to 6:00 PM, Monday through Friday**. For urgent support outside these hours, feel free to email us—we're here to help whenever you need! 😊

---

---
## Pattern 3 — Parallelization

**Concept:** Run multiple independent LLM calls at the same time and aggregate the results. Two sub-patterns:

- **Sectioning** — split a task into independent chunks (e.g. evaluate different aspects of code).
- **Voting** — run the same prompt N times and take a majority decision.

```
               ┌─ [Worker 1] ─┐
Input ─────────┤─ [Worker 2] ─├──► Aggregator ──► Final output
               └─ [Worker 3] ─┘
```

**Use when:** subtasks are independent, or you need higher confidence via multiple perspectives.

**Example below (Sectioning):** Evaluate a code snippet on three independent axes simultaneously — correctness, readability, security.


In [4]:
CODE_SNIPPET = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query).fetchone()
"""

REVIEW_AXES = {
    "correctness":  "Review the code only for logical correctness and potential bugs.",
    "readability":  "Review the code only for readability, naming, and documentation.",
    "security":     "Review the code only for security vulnerabilities.",
}

def review_axis(axis: str, instructions: str) -> tuple[str, str]:
    result = chat([
        {"role": "system", "content": instructions + " Be concise (3–5 sentences)."},
        {"role": "user",   "content": f"```python\n{CODE_SNIPPET}\n```"},
    ])
    return axis, result

# ── Run reviews in parallel using threads ─────────────────────────────────────
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(review_axis, axis, instr): axis
               for axis, instr in REVIEW_AXES.items()}
    reviews = {}
    for future in futures:
        axis, result = future.result()
        reviews[axis] = result

# ── Aggregation step ──────────────────────────────────────────────────────────
review_text = "\n\n".join(f"### {k.title()}\n{v}" for k, v in reviews.items())
display(Markdown("## Parallel Code Reviews\n\n" + review_text))

summary = chat([
    {"role": "system", "content": "You are a lead code reviewer. Summarise the findings below into a single priority-ordered action list."},
    {"role": "user",   "content": review_text},
])
display(Markdown("## Aggregated Action List\n\n" + summary))


## Parallel Code Reviews

### Correctness
The code is vulnerable to SQL injection due to direct string formatting of `user_id` into the query. Always use parameterized queries (e.g., `execute(query, (user_id,))`) to prevent malicious input. Additionally, the function lacks error handling for database exceptions, which could mask issues during execution.

### Readability
The function lacks security by using string interpolation for SQL queries, making it vulnerable to SQL injection. Rename `query` to `sql_query` for clarity. Add a docstring to explain the function's purpose and parameters. Consider using parameterized queries (`db.execute(query, (user_id,))`) for safer database interactions.

### Security
The code is vulnerable to SQL injection due to the direct concatenation of `user_id` into the query string. Using string formatting (`f"..."`) allows attackers to inject malicious SQL via input like `1 OR 1=1`, bypassing authentication. Parameterized queries (e.g., using `?` or `execute()` with a tuple) should replace the string interpolation to safely handle user input. No input validation or sanitization is performed, further exacerbating the risk.

## Aggregated Action List

### Priority-Ordered Action List

1. **Fix SQL Injection Vulnerability**  
   - Replace direct string formatting (e.g., `f"SELECT * FROM users WHERE id = {user_id}"`) with **parameterized queries** (e.g., `db.execute("SELECT * FROM users WHERE id = ?", (user_id,))`).  
   - Use placeholders (`?` or named parameters) instead of string interpolation to prevent malicious input exploitation.  

2. **Add Database Exception Handling**  
   - Implement try-except blocks to catch and log database exceptions (e.g., `try: ... except db.Error as e: log.error(e)`). This ensures errors are not silently ignored and aids in debugging.  

3. **Improve Readability and Documentation**  
   - Rename the `query` variable to `sql_query` for clarity.  
   - Add a docstring to the function to describe its purpose, parameters, and expected behavior.  

4. **Optional: Input Validation (Secondary Security Measure)**  
   - Validate and sanitize `user_id` input (e.g., check it is an integer or UUID) to further mitigate risks, though parameterized queries already address the primary SQL injection vulnerability.  

---  
**Rationale:** Security fixes (SQL injection) are critical and must be prioritized. Correctness (error handling) and readability (docstrings/variable names) follow as essential improvements. Input validation is optional but recommended for enhanced security.

### Parallelization — Voting variant

Run the same classification N times and take a majority vote. Useful when a single call may be unreliable.


In [5]:
from collections import Counter

REVIEW_TEXT = """
The new product update is amazing! I love how smooth everything feels now.
Although the onboarding is still a bit confusing, the core experience is 10/10.
"""

def classify_sentiment(_: int) -> str:
    """Classify sentiment as positive, negative, or mixed."""
    return chat(
        [
            {"role": "system", "content": "Classify the sentiment as exactly one word: positive, negative, or mixed."},
            {"role": "user",   "content": REVIEW_TEXT},
        ],
        temperature=1,   # higher temp → more variance across runs
    ).strip().lower().split()[0]

N_VOTES = 5
with ThreadPoolExecutor(max_workers=N_VOTES) as pool:
    votes = list(pool.map(classify_sentiment, range(N_VOTES)))

tally = Counter(votes)
winner = tally.most_common(1)[0][0]
display(Markdown(f"**Votes:** {dict(tally)}\n\n**Majority verdict:** `{winner}`"))


**Votes:** {'mixed': 5}

**Majority verdict:** `mixed`

---
## Pattern 4 — Orchestrator-Workers

**Concept:** A central *orchestrator* LLM receives the goal, breaks it into subtasks dynamically, and delegates each subtask to a *worker* LLM. The orchestrator then synthesises all worker outputs.

```
Goal ──► [Orchestrator LLM] ──► subtask_1 ──► [Worker 1]
                             ├─► subtask_2 ──► [Worker 2]   ──► [Orchestrator synthesises]
                             └─► subtask_3 ──► [Worker 3]
```

**Key difference from Parallelization:** subtasks are *not pre-defined*; the orchestrator decides them at runtime.

**Use when:** the task is complex and the required subtasks aren't known in advance.

**Example below:** Research assistant — the orchestrator decides which sub-questions to investigate, workers answer each, orchestrator synthesises a final report.


In [6]:
RESEARCH_GOAL = "Explain how Retrieval-Augmented Generation (RAG) works and when to use it."

# ── Orchestrator: break into subtasks ─────────────────────────────────────────
plan_raw = chat(
    [
        {"role": "system", "content": (
            "You are a research orchestrator. Given a goal, output a JSON array of 3–4 "
            "focused sub-questions to investigate. Return ONLY valid JSON, no other text."
        )},
        {"role": "user", "content": RESEARCH_GOAL},
    ],
    temperature=0,
)

# Parse JSON (strip markdown fences if the model adds them)
plan_clean = plan_raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
subtasks: list[str] = json.loads(plan_clean)
display(Markdown("### Orchestrator Plan\n\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(subtasks))))

# ── Workers: answer each subtask in parallel ──────────────────────────────────
def worker_answer(question: str) -> tuple[str, str]:
    answer = chat([
        {"role": "system", "content": "You are a concise AI/ML expert. Answer in 3–5 sentences."},
        {"role": "user",   "content": question},
    ])
    return question, answer

with ThreadPoolExecutor(max_workers=len(subtasks)) as pool:
    worker_results = dict(pool.map(lambda q: worker_answer(q), subtasks))

worker_text = "\n\n".join(f"**Q: {q}**\n\n{a}" for q, a in worker_results.items())
display(Markdown("### Worker Answers\n\n" + worker_text))

# ── Orchestrator: synthesise into a final report ──────────────────────────────
final_report = chat([
    {"role": "system", "content": "You are a senior technical writer. Synthesise the Q&A pairs below into a coherent, structured mini-report of ~200 words."},
    {"role": "user",   "content": f"Goal: {RESEARCH_GOAL}\n\n{worker_text}"},
])
display(Markdown("### Final Synthesised Report\n\n" + final_report))


### Orchestrator Plan

1. {'question': 'How does Retrieval-Augmented Generation (RAG) integrate retrieval and generation processes to produce responses?'}
2. {'question': 'What are the key components of a RAG system, and how do they interact?'}
3. {'question': 'In what scenarios is RAG more effective than traditional models like pure retrieval or generation systems?'}

BadRequestError: Error code: 400 - {'error': {'message': 'invalid message content type: map[string]interface {}', 'type': 'invalid_request_error', 'param': None, 'code': None}}

---
## Pattern 5 — Evaluator-Optimizer

**Concept:** A *generator* LLM produces output; an *evaluator* LLM scores it and provides feedback; the generator uses that feedback to improve. The loop continues until quality criteria are met or a maximum iteration count is reached.

```
Task ──► [Generator] ──► draft
                ▲              │
           feedback      [Evaluator]
                ▲              │
                └──── score ◄──┘
           (loop until score ≥ threshold or max_iters)
```

**Use when:** you have clear quality criteria and iterative refinement measurably improves output. Analogous to a human writer → editor cycle.

**Example below:** Refine a technical explanation until the evaluator scores it ≥ 8/10 for clarity.


In [ ]:
WRITING_TASK = "Explain how attention mechanisms work in transformer models, for a junior developer."
SCORE_THRESHOLD = 8
MAX_ITERATIONS  = 4

def generate(task: str, feedback: str | None = None) -> str:
    messages = [
        {"role": "system", "content": "You are a technical educator. Write clearly and concisely (≤150 words)."},
        {"role": "user",   "content": task},
    ]
    if feedback:
        messages.append({"role": "user", "content": f"Improve your previous response based on this feedback:\n{feedback}"})
    return chat(messages)

def evaluate(task: str, response: str) -> tuple[int, str]:
    """Return (score 1-10, actionable feedback)."""
    raw = chat(
        [
            {"role": "system", "content": (
                "You are a strict writing evaluator. "
                "Score the response from 1–10 for clarity and accuracy given the task. "
                "Reply ONLY with JSON: {\"score\": <int>, \"feedback\": \"<string>\"}"
            )},
            {"role": "user", "content": f"Task: {task}\n\nResponse:\n{response}"},
        ],
        temperature=0,
    ).strip()
    clean = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    parsed = json.loads(clean)
    return int(parsed["score"]), parsed["feedback"]

# ── Optimization loop ─────────────────────────────────────────────────────────
feedback = None
for iteration in range(1, MAX_ITERATIONS + 1):
    draft = generate(WRITING_TASK, feedback)
    score, feedback = evaluate(WRITING_TASK, draft)

    display(Markdown(
        f"### Iteration {iteration} — Score: {score}/10\n\n"
        f"**Draft:**\n\n{draft}\n\n"
        f"**Evaluator feedback:** {feedback}"
    ))

    if score >= SCORE_THRESHOLD:
        display(Markdown(f"✅ **Threshold reached at iteration {iteration}. Final answer above.**"))
        break
else:
    display(Markdown(f"⚠️ Max iterations reached. Best score: {score}/10"))


---
## Pattern Comparison

Run the same task through **Prompt Chaining** vs **Evaluator-Optimizer** and compare quality vs latency trade-offs.


In [ ]:
import time

TASK = "Write a 100-word explanation of embeddings for a product manager."

# ── Prompt Chaining: outline → draft (2 steps, no feedback loop) ──────────────
t0 = time.time()
outline_pm = chat([
    {"role": "system", "content": "You are a technical content strategist."},
    {"role": "user",   "content": f"Write a 3-point outline for: {TASK}"},
])
draft_pm = chat([
    {"role": "system",    "content": "You are a technical writer for business audiences."},
    {"role": "assistant", "content": outline_pm},
    {"role": "user",      "content": "Expand into a 100-word explanation."},
])
chain_time = time.time() - t0

# ── Evaluator-Optimizer: 1 generation + 1 refinement pass ─────────────────────
t0 = time.time()
draft_eo = generate(TASK)
score_eo, fb_eo = evaluate(TASK, draft_eo)
if score_eo < 8:
    draft_eo = generate(TASK, fb_eo)
eo_time = time.time() - t0

display(Markdown(
    f"## Prompt Chaining result ({chain_time:.1f}s)\n\n{draft_pm}\n\n---\n\n"
    f"## Evaluator-Optimizer result ({eo_time:.1f}s) — initial score {score_eo}/10\n\n{draft_eo}"
))


---
## Exercises

Work through as many as you like. Start with (1) and go deeper as time permits.

1. **Prompt Chaining - add retry + structured gate:** In the blog pipeline (Pattern 1), enforce a strict outline format (exactly 3 numbered sections with titles only). If the gate fails, automatically re-prompt the model up to 2 retries before halting. Print which attempt passed.

2. **Routing — add a fourth category:** Extend the routing example to handle `"feature_request"` queries. Write a specialist prompt and test it with at least two example inputs.

3. **Parallelization — build a guardrail:** Create a parallel two-worker setup where Worker A generates a response to a user question while Worker B simultaneously checks whether the question is safe/appropriate. Combine the two results — only return Worker A's output if Worker B gives the green light.

4. **Orchestrator-Workers — dynamic depth:** Modify the orchestrator example so that after the first round of worker answers, the orchestrator decides whether a second round of deeper sub-questions is warranted. Implement at most 2 rounds.

5. **Evaluator-Optimizer — custom rubric:** Change the evaluator prompt to score on three separate axes (clarity, accuracy, brevity) and return a JSON object with all three scores. Stop the loop only when *all three* scores are ≥ 7.

6. **(Stretch) Combine patterns:** Build a mini content pipeline that uses:
   - **Routing** to detect if the topic is technical or business-oriented,
   - **Orchestrator-Workers** to research the topic in parallel, and
   - **Evaluator-Optimizer** to refine the final output.


---
## Exercise#1 -Prompt Chaining 
with retry logic and structured gates

In [12]:
TOPIC = "Stock Investment for Dummies"

def generate_outline_with_retries(topic, max_retries=2):
    """Generate a strict 3-section outline with retry logic."""
    attempt = 0
    outline = None
    
    while attempt <= max_retries:
        attempt += 1
        outline = chat([
            {"role": "system", "content": "You are a technical content strategist."},
            {"role": "user",   "content": f"Write an outline for a blog post about: {topic}. "
                                          "Format: exactly 3 numbered sections (1., 2., 3.) with titles only. "
                                          "No extra text, no subsections."},
        ])
        
        # Gate check: must have exactly 3 numbered sections
        sections = re.findall(r'^\d+\.\s+.+$', outline, flags=re.MULTILINE)
        gate_pass = (len(sections) == 3)
        
        print(f"🔍 Attempt {attempt} — Gate pass: {gate_pass}")
        
        if gate_pass:
            print(f"✅ Outline passed on attempt {attempt}")
            return outline
    
    # If all retries fail
    raise ValueError("❌ Outline generation failed after retries.")

# ── Step 1: Generate outline with retry ───────────────────────────────────────
outline = generate_outline_with_retries(TOPIC)
display(Markdown("### Step 1 — Outline\n\n" + outline))

# ── Step 2: Write the draft ───────────────────────────────────────────────────
draft = chat([
    {"role": "system",    "content": "You are a senior technical writer."},
    {"role": "user",      "content": f"Topic: {TOPIC}"},
    {"role": "assistant", "content": outline},
    {"role": "user",      "content": "Expand the outline into a concise 200-word blog post draft."},
])
display(Markdown("### Step 2 — Draft\n\n" + draft))

# ── Step 3: Polish for a non-technical audience ───────────────────────────────
polished = chat([
    {"role": "system", "content": "You are an editor who simplifies technical writing."},
    {"role": "user",   "content": f"Rewrite the following blog post so it is engaging for a non-technical audience. "
                                  f"Keep it under 150 words.\n\n{draft}"},
])
display(Markdown("### Step 3 — Polished\n\n" + polished))


🔍 Attempt 1 — Gate pass: True
✅ Outline passed on attempt 1


### Step 1 — Outline

1. Understanding the Basics of Stock Investment  
2. How to Start Investing in Stocks  
3. Common Mistakes to Avoid as a Beginner

### Step 2 — Draft

**Stock Investment for Dummies: A Beginner’s Guide**  

Stock investing can feel overwhelming, but it’s simpler than you think. At its core, a stock represents ownership in a company. When you buy shares, you become a partial owner, potentially benefiting from the company’s growth and profits. However, stock markets can be volatile, so understanding risks and rewards is key.  

To start, choose a brokerage platform that suits your needs—whether you prefer low fees, mobile access, or research tools. Open a brokerage account, fund it, and begin by researching companies or indices that align with your goals. Start small, focusing on a few stocks or ETFs to build confidence.  

Avoid common pitfalls: don’t let emotions drive decisions, like panic selling during dips. Instead, stick to a long-term strategy. Also, avoid overtrading, which can erode profits. Diversify your portfolio to spread risk. Remember, consistent learning and patience are your greatest allies. With the right approach, stock investing can be a rewarding journey—no expertise required!  

*(Word count: 200)*

### Step 3 — Polished

**Stock Investing Made Simple: A Beginner’s Guide**  

Investing in stocks might seem scary, but it’s easier than you think! A stock is like owning a tiny piece of a company. When it grows, you could too—though markets can swing up and down.  

Start by picking a brokerage that fits your needs (low fees, easy access, or research tools). Open an account, fund it, and begin with a few stocks or exchange-traded funds (ETFs) to keep things simple.  

Avoid letting fear or excitement control your decisions. Stick to a plan, avoid overtrading, and spread your investments to lower risks. Learning and patience are your best tools. With the right approach, stock investing can be a smart, rewarding journey—no need for expertise!  

*(Word count: 149)*

---
## Exercise#2 -Routing 
with additional feature request

In [ ]:
SPECIALIST_PROMPTS = {
    "billing":   "You are a billing support specialist. Be empathetic and solution-focused.",
    "technical": "You are a senior technical support engineer. Give precise, step-by-step answers.",
    "general":   "You are a friendly customer success agent. Be warm and concise.",
    "feature_request": "You are a Financial Manager. Evaluate feature requests in terms of cost, ROI, and budget impact. "
                       "Explain potential financial value and suggest prioritization based on resource allocation.",
}

def classify_query(user_query: str) -> str:
    """Return one of: billing | technical | general | feature_request"""
    result = chat(
        [
            {"role": "system", "content": (
                "Classify the following customer query into exactly one category: "
                "billing, technical, general, or feature_request. "
                "Reply with the single word only."
            )},
            {"role": "user", "content": user_query},
        ],
        temperature=0,
    ).strip().lower()
    # Normalize — model might add punctuation
    for key in SPECIALIST_PROMPTS:
        if key in result:
            return key
    return "general"

def routed_response(user_query: str):
    category = classify_query(user_query)
    system_prompt = SPECIALIST_PROMPTS[category]
    answer = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query},
    ])
    return category, answer

# ── Test queries ──────────────────────────────────────────────────────────────
queries = [
    "I was charged twice on my last invoice.",   # billing
    "My API requests keep returning a 429 error.", # technical
    "What are your business hours?",             # general
    "Can you add offline support for the mobile app?", # feature_request
    "I’d like a dark mode feature in the app.",  # feature_request
]

for q in queries:
    category, answer = routed_response(q)
    display(Markdown(f"**Query:** {q}\n\n**Routed to:** `{category}`\n\n**Response:** {answer}\n\n---"))

**Query:** I was charged twice on my last invoice.

**Routed to:** `billing`

**Response:** I'm sorry to hear you encountered this issue—it's frustrating to see duplicate charges. Let me help resolve this for you. Could you share the invoice number(s) or the dates of the charges? This will help me locate the specific transaction and determine the best course of action. 

If you're unsure of the invoice numbers, I can guide you through checking your account for duplicate charges. Once I have the details, I'll work with our billing team to investigate and ensure the extra charge is corrected, whether that means a refund or preventing future duplicates. Let me know how I can assist further!

---

**Query:** My API requests keep returning a 429 error.

**Routed to:** `technical`

**Response:** Here's a step-by-step guide to resolve **429 Too Many Requests** errors:

---

### **1. Confirm Rate Limiting is the Cause**
- **429** means the server is throttling your requests due to exceeding rate limits.
- Check API documentation for **rate limit policies** (e.g., requests/minute, requests/hour).

---

### **2. Monitor Request Frequency**
- **Slow down requests**: Ensure your application isn't sending requests faster than allowed.
- **Use tools**: Log request timestamps to verify if you're hitting the limit.

---

### **3. Check API Key/IP Address**
- **Rate limits are often per API key or IP**:  
  - If you're using the same API key across multiple IPs (e.g., via proxies), each IP might be hitting its own limit.
  - Ensure all requests use the **same API key** (if required) and are made from a **single IP**.

---

### **4. Review API Documentation for Rate Limit Headers**
- Many APIs include headers like:
  - `X-RateLimit-Remaining` (requests left)
  - `X-RateLimit-Reset` (timestamp when the limit resets)
- Example:  
  ```bash
  curl -I https://api.example.com/endpoint
  ```
  Check headers for rate limit info.

---

### **5. Implement Retry Logic with Exponential Backoff**
- Use **exponential backoff** to retry requests after delays (e.g., 1s, 4s, 16s).
- Example in Python:
  ```python
  import time
  import requests

  for attempt in range(5):
      try:
          response = requests.get(url, headers=headers)
          response.raise_for_status()
          break
      except requests.exceptions.HTTPError as e:
          if response.status_code == 429:
              wait = 2 ** attempt  # Exponential backoff
              time.sleep(wait)
          else:
              raise
  ```

---

### **6. Use a Rate Limiter in Your Code**
- Implement a **token bucket** or **sliding window** rate limiter to control request frequency.
- Libraries like `ratelimit` (Python) or `express-rate-limit` (Node.js) can help.

---

### **7. Check for Third-Party Tools or Libraries**
- If using a library (e.g., SDKs), confirm it's not generating excessive requests (e.g., polling or retries).

---

### **8. Contact API Provider Support**
- If you're certain you're within limits:
  - Provide **request logs** (timestamps, IPs, API keys).
  - Ask about **temporary rate limit increases** or **account-specific limits**.

---

### **9. Upgrade Your Plan (if applicable)**
- Some APIs offer **higher rate limits** for paid tiers. Check if your plan matches your usage.

---

### **10. Cache Responses (if applicable)**
- If the API allows caching, store responses to reduce redundant requests.

---

**Summary**:  
- Slow requests, verify API key/IP, use retry logic, and check rate limit headers.  
- If unresolved, contact the API provider with detailed logs.

---

**Query:** What are your business hours?

**Routed to:** `general`

**Response:** We're here to help! Our business hours are **Monday to Friday, 8 AM to 6 PM**. For urgent issues outside these hours, we’re available **24/7** to assist with critical matters. Let me know how I can help! 😊

---

**Query:** Can you add offline support for the mobile app?

**Routed to:** `feature_request`

**Response:** **Evaluation of Offline Support Feature Request**  

**1. Cost Analysis**  
- **Development Cost**:  
  - Backend: Implement data sync mechanisms (e.g., conflict resolution, versioning), server-side logic for offline-to-online sync, and storage optimization.  
  - Frontend: Integrate local caching (e.g., SQLite, IndexedDB), handle offline state management, and ensure seamless transitions between online/offline modes.  
  - Testing: Additional QA for edge cases (e.g., partial sync, data corruption).  
  - **Estimated Cost**: $50,000–$100,000 (depending on complexity and team size).  

- **Maintenance Cost**:  
  - Ongoing updates for OS compatibility, bug fixes, and scalability as user adoption grows.  

**2. ROI Potential**  
- **User Experience**:  
  - Reduces app abandonment in low-connectivity scenarios, improving retention and satisfaction.  
  - Potential to increase user engagement by enabling critical tasks (e.g., data entry, reports) without internet.  
- **Business Impact**:  
  - If the app serves industries with unreliable connectivity (e.g., field services, logistics), this could directly reduce operational downtime and support costs.  
  - Mitigates revenue loss from users unable to access core features.  
- **Competitive Edge**:  
  - Positions the app as more robust than competitors lacking offline support, potentially driving market share growth.  

**3. Budget Impact**  
- **Short-Term**:  
  - Could divert resources from other projects (e.g., new features, security upgrades).  
  - May require reallocating existing budget or requesting additional funding.  
- **Long-Term**:  
  - Likely to yield cost savings through reduced support tickets and increased user loyalty.  

**4. Prioritization Recommendations**  
- **High Priority**:  
  - If the user base includes segments with poor connectivity (e.g., rural users, field workers) or if connectivity issues are a major support pain point.  
  - If the app’s core functionality relies on offline access (e.g., data collection, real-time reporting).  
- **Medium Priority**:  
  - If the user base primarily has stable internet, but there is potential for future expansion into low-connectivity markets.  
- **Low Priority**:  
  - If the app is consumer-facing with minimal reliance on offline capabilities and no immediate business-critical use cases.  

**5. Financial Value Summary**  
- **Direct Value**:  
  - Reduced support costs, higher user retention, and potential revenue growth from expanded user access.  
- **Indirect Value**:  
  - Enhanced brand reputation as a reliable tool, which could lead to upsell opportunities or partnerships.  

**6. Suggested Approach**  
- **Phase 1**: Conduct a cost-benefit analysis to quantify current support costs tied to connectivity issues.  
- **Phase 2**: Pilot the feature with a small user group to validate ROI and refine requirements.  
- **Phase 3**: Allocate budget and resources based on prioritization, balancing this request with other strategic initiatives.  

**Final Recommendation**:  
Prioritize this request if it aligns with critical user needs or business goals (e.g., expanding into new markets). Allocate resources incrementally to minimize upfront costs while maximizing long-term ROI. If budgets are constrained, consider a phased rollout or hybrid approach (e.g., core offline features first).

---

**Query:** I’d like a dark mode feature in the app.

**Routed to:** `feature_request`

**Response:** **Evaluation of Dark Mode Feature Request**  
**Financial Manager Perspective**  

---

### **1. Cost Analysis**  
- **Development Cost**:  
  - **Scope**: Implementing dark mode requires modifying UI elements (buttons, text, backgrounds), ensuring cross-platform compatibility (iOS/Android), and testing for visual consistency.  
  - **Effort**: Moderate to high, depending on app complexity. For a standard app, this could take **2–4 weeks** of developer time.  
  - **Third-Party Tools**: If using frameworks like Flutter or React Native, reusable components may reduce effort.  

- **Testing Cost**:  
  - **QA Overhead**: Additional testing cycles to ensure dark mode doesn’t break existing functionality or cause accessibility issues (e.g., contrast ratios).  

- **Ongoing Maintenance**:  
  - Minor updates may be needed for future OS updates or design changes.  

---

### **2. ROI and Financial Value**  
- **User Retention & Satisfaction**:  
  - Dark mode improves user experience, especially for nighttime use, potentially increasing retention and reducing churn.  
  - Studies show users prefer dark mode for reduced eye strain, which could lower support costs related to user complaints.  

- **Market Differentiation**:  
  - Competitors may already offer dark mode; this could help maintain competitiveness but may not guarantee a revenue lift.  

- **App Store Visibility**:  
  - While not directly tied to rankings, a visually appealing UI (including dark mode) may indirectly improve app store impressions.  

- **Accessibility Compliance**:  
  - Dark mode can be part of accessibility features, reducing legal risks and improving inclusivity.  

---

### **3. Budget Impact**  
- **One-Time vs. Ongoing Costs**:  
  - Development and testing are one-time costs. Ongoing maintenance is minimal.  
  - **Budget Allocation**: If the app has a dedicated development budget, this is feasible. If not, prioritize based on user demand.  

- **Alternative Cost-Saving Opportunities**:  
  - Reuse existing UI components or leverage open-source libraries (e.g., Material Design) to cut development time.  

---

### **4. Prioritization & Resource Allocation**  
- **High Priority (if):**  
  - User feedback strongly supports dark mode (e.g., surveys, analytics).  
  - Target audience (e.g., night-shift workers, low-light users) benefits significantly.  
  - Competitors have dark mode, and users expect it as a standard feature.  

- **Medium Priority (if):**  
  - Limited user demand, but the feature aligns with long-term UI/UX goals.  
  - Resource allocation is constrained, and other features (e.g., performance optimizations) are more urgent.  

- **Low Priority (if):**  
  - Minimal user demand and no clear ROI.  
  - Development resources are better spent on high-impact features (e.g., new revenue streams, critical bug fixes).  

---

### **5. Recommendations**  
- **Phase 1 (Quick Win)**:  
  - Implement a **lightweight dark mode toggle** using existing UI components to test user adoption without major rework.  
  - Allocate **2–3 weeks** of developer time and **$5,000–$10,000** (depending on team rates).  

- **Phase 2 (Full Rollout)**:  
  - Expand to full dark mode with accessibility checks and cross-platform testing.  
  - Allocate **4–6 weeks** and **$15,000–$30,000**.  

- **ROI Metrics to Track**:  
  - User retention rates, app store ratings, and support ticket volume post-launch.  

- **Risk Mitigation**:  
  - Conduct A/B testing to measure user preference.  
  - Ensure dark mode adheres to accessibility standards (e.g., WCAG contrast ratios).  

---

### **Conclusion**  
Dark mode offers **moderate to high ROI** through improved user experience and retention, but its **budget impact depends on development scope**. Prioritize it if user demand is strong and resources are available, or test it as a low-risk MVP to validate value before full investment.

---

In [13]:
SPECIALIST_PROMPTS = {
    "billing":   "You are a billing support specialist. Be empathetic and solution-focused.",
    "technical": "You are a senior technical support engineer. Give precise, step-by-step answers.",
    "general":   "You are a friendly customer success agent. Be warm and concise.",
    "feature_request": "You are a Financial Manager. Evaluate feature requests in terms of cost, ROI, and budget impact. "
                       "Explain potential financial value and suggest prioritization based on resource allocation.",
}

def classify_query(user_query: str) -> str:
    """Return one of: billing | technical | general | feature_request"""
    result = chat(
        [
            {"role": "system", "content": (
                "Classify the following customer query into exactly one category: "
                "billing, technical, general, or feature_request. "
                "Reply with the single word only."
            )},
            {"role": "user", "content": user_query},
        ],
        temperature=0,
    ).strip().lower()
    # Normalize — model might add punctuation
    for key in SPECIALIST_PROMPTS:
        if key in result:
            return key
    return "general"

def routed_response(user_query: str):
    category = classify_query(user_query)
    system_prompt = SPECIALIST_PROMPTS[category]
    answer = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query},
    ])
    return category, answer

# ── Unit Tests ────────────────────────────────────────────────────────────────
def test_routing():
    # Billing query
    q1 = "I was charged twice on my last invoice."
    assert classify_query(q1) == "billing"

    # Technical query
    q2 = "My API requests keep returning a 429 error."
    assert classify_query(q2) == "technical"

    # General query
    q3 = "What are your business hours?"
    assert classify_query(q3) == "general"

    # Feature request queries
    q4 = "Can you add offline support for the mobile app?"
    assert classify_query(q4) == "feature_request"

    q5 = "I’d like a dark mode feature in the app."
    assert classify_query(q5) == "feature_request"

    print("✅ All routing tests passed!")

# Run the tests
test_routing()

✅ All routing tests passed!


---
## Exercise#3 -Parallelization 
with guardrail


In [ ]:

def worker_a_generate(user_query: str) -> str:
    """Worker A: Generate a response to the user query."""
    response = chat([
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_query},
    ])
    return response

def worker_b_guardrail(user_query: str) -> bool:
    """Worker B: Check if the query is safe/appropriate."""
    verdict = chat([
        {"role": "system", "content": (
            "Decide if the following query is safe and appropriate. "
            "Reply only with 'safe' or 'unsafe'."
        )},
        {"role": "user", "content": user_query},
    ]).strip().lower()
    return verdict == "safe"

def guarded_response(user_query: str):
    """Run Worker A and Worker B in parallel, return A's output only if B approves."""
    with ThreadPoolExecutor() as executor:
        future_a = executor.submit(worker_a_generate, user_query)
        future_b = executor.submit(worker_b_guardrail, user_query)
        response_a = future_a.result()
        verdict_b = future_b.result()
    if verdict_b:
        return f"✅ Safe. Response:\n{response_a}"
    else:
        return "❌ Query deemed unsafe. No response generated."

# ── Example usage ──────────────────────────────────────────────────────────
queries = [
    "Can you explain how vector databases help retrieval-augmented generation?",
    "Tell me how to hack into someone's account.",  # unsafe
]

for q in queries:
    print(f"User query: {q}")
    print(guarded_response(q))
    print("---")


## Exercise 3: Parallelization


User query: Can you explain how vector databases help retrieval-augmented generation?
✅ Safe. Response:
Vector databases play a crucial role in **retrieval-augmented generation (RAG)** by enabling efficient and accurate retrieval of relevant information from large datasets. Here's how they contribute to RAG:

---

### **1. Semantic Search via Vector Embeddings**
- **Vector Representation**: In RAG, documents or data are converted into **vector embeddings** (numerical representations) using models like BERT, DPR, or other NLP techniques. These vectors capture semantic meaning (e.g., "dog" and "animal" are semantically similar).
- **Similarity-Based Retrieval**: Vector databases use **approximate nearest neighbor (ANN)** algorithms (e.g., HNSW, FAISS, or Annoy) to quickly find vectors in the database that are most similar to a query's embedding. This allows **semantic similarity** rather than keyword-based matching, ensuring relevant information is retrieved even for complex or ambiguous

---
## Exercise#4- Orchestrator-Workers 
with dynamic depth

In [7]:
import json
from concurrent.futures import ThreadPoolExecutor
from IPython.display import Markdown
 
RESEARCH_GOAL = "Explain how Stocks Investment works and when the right time to invest."

# ── Utility: clean JSON output from model ─────────────────────────────────────
def clean_json(raw: str) -> str:
    text = raw.strip()
    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    if text.startswith("```"):
        text = text[len("```"):].strip()
    if text.endswith("```"):
        text = text[:-3].strip()
    return text

# ── Orchestrator: generate subtasks ──────────────────────────────────────────
def orchestrator_plan(goal: str, context: str = "", max_retries: int = 2) -> list[str]:
    """Generate subtasks based on the goal and optional context (previous answers)."""
    system_prompt = (
        "You are a research orchestrator. Given a goal, output ONLY a JSON array of 3–4 "
        "focused sub-questions (strings). Do not add explanations or text outside JSON."
    )
    if context:
        user_prompt = f"Goal: {goal}\n\nContext from previous answers:\n{context}"
    else:
        user_prompt = goal

    for attempt in range(max_retries):
        plan_raw = chat([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ], temperature=0)

        plan_clean = clean_json(plan_raw)
        try:
            subtasks = json.loads(plan_clean)
            if isinstance(subtasks, list) and all(isinstance(t, str) for t in subtasks):
                return subtasks
        except json.JSONDecodeError:
            print(f"⚠️ Invalid JSON on attempt {attempt+1}, retrying...")

    # Fallback if all retries fail
    print("⚠️ Using fallback subtasks.")
    return [
        f"What is {goal}?",
        f"How does {goal} work in practice?",
        f"When should {goal} be applied?",
    ]

# ── Worker: answer a subtask ─────────────────────────────────────────────────
def worker_answer(question: str) -> tuple[str, str]:
    answer = chat([
        {"role": "system", "content": "You are a concise AI/ML expert. Answer in 3–5 sentences."},
        {"role": "user",   "content": question},
    ])
    return question, answer

# ── Orchestrator: decide if deeper round is needed ───────────────────────────
def orchestrator_decide(goal: str, worker_text: str) -> bool:
    decision = chat([
        {"role": "system", "content": (
            "You are a research orchestrator. Given the goal and the first round of answers, "
            "decide if a second round of deeper sub-questions is warranted. "
            "Reply ONLY with 'yes' or 'no'."
        )},
        {"role": "user", "content": f"Goal: {goal}\n\nAnswers:\n{worker_text}"},
    ], temperature=0).strip().lower()
    return decision == "yes"

# ── Pipeline ─────────────────────────────────────────────────────────────────
rounds = []
goal = RESEARCH_GOAL
context = ""

for depth in range(1, 3):  # at most 2 rounds
    try:
        subtasks = orchestrator_plan(goal, context)
    except ValueError as e:
        print(f"⚠️ Orchestrator failed to generate subtasks: {e}")
        break

    if not subtasks:
        print("⚠️ No subtasks generated, stopping pipeline.")
        break

    # Numbering starts at 1 for readability
    display(Markdown(
        f"### Round {depth} — Orchestrator Plan\n\n" +
        "\n".join(f"{i}. {t}" for i, t in enumerate(subtasks, start=1))
    ))

    # Run workers in parallel
    with ThreadPoolExecutor(max_workers=len(subtasks)) as pool:
        worker_results = list(pool.map(worker_answer, subtasks))

    # Build worker answers text
    worker_text = "\n\n".join(f"**Q: {q}**\n\n{a}" for q, a in worker_results)
    display(Markdown(f"### Round {depth} — Worker Answers\n\n" + worker_text))

    rounds.append(worker_text)

    # Decision logic: after first round, orchestrator decides if deeper round is needed
    if depth == 1:
        if not orchestrator_decide(goal, worker_text):
            break
        else:
            # Pass previous answers as context for deeper subtasks
            context = worker_text

# ── Final synthesis ─────────────────────────────────────────────────────────
combined_text = "\n\n".join(rounds)
final_report = chat([
    {"role": "system", "content": (
        "You are a senior financial advisor. Synthesise the Q&A pairs below into a coherent, "
        "structured mini-report of ~200 words."
    )},
    {"role": "user", "content": f"Goal: {RESEARCH_GOAL}\n\n{combined_text}"},
])
display(Markdown("### Final Synthesised Report\n\n" + final_report))


### Round 1 — Orchestrator Plan

1. What are stocks and how do they represent ownership in a company?
2. What factors influence the right time to invest in stocks?
3. What strategies can help determine optimal investment timing?
4. How do market conditions and economic indicators affect stock investment decisions?

### Round 1 — Worker Answers

**Q: What are stocks and how do they represent ownership in a company?**

Stocks represent ownership in a company through shares, with each share granting proportional ownership of the company's assets and earnings. Shareholders elect the board of directors and may vote on major decisions. They can receive dividends from profits and benefit from stock price appreciation. Ownership stakes are determined by the number of shares held, though stock prices fluctuate based on market demand and company performance.

**Q: What factors influence the right time to invest in stocks?**

The right time to invest in stocks depends on market conditions (bull/bear phases), economic indicators (interest rates, GDP), company fundamentals (earnings, growth), valuation metrics (P/E ratios), and investor sentiment. Long-term goals, risk tolerance, and diversification also play critical roles. Timing the market is challenging, so many experts recommend consistent investing over trying to predict short-term fluctuations. External factors like geopolitical events or industry trends can further influence optimal investment windows.

**Q: What strategies can help determine optimal investment timing?**

Optimal investment timing can be determined through fundamental analysis (evaluating financial metrics and economic indicators), technical analysis (using price trends and chart patterns), and risk management strategies (like diversification and stop-loss orders). Monitoring market sentiment and macroeconomic trends also helps identify potential entry points. Consistent dollar-cost averaging can mitigate timing risks, while aligning investments with personal financial goals and risk tolerance ensures long-term alignment. However, market timing is inherently challenging, so a balanced approach combining discipline and adaptability is key.

**Q: How do market conditions and economic indicators affect stock investment decisions?**

Market conditions and economic indicators shape stock investment decisions by signaling economic health and future trends. Strong GDP growth, low unemployment, and rising corporate profits often boost investor confidence, driving stock prices up. Conversely, high inflation, rising interest rates, or recessions may prompt caution, leading to risk-off behavior. Investors use indicators like PMIs, CPI, and Fed policy statements to time entries/ exits, though market sentiment and geopolitical factors also play critical roles. While these signals guide decisions, they are not perfect predictors, requiring diversification and risk management.

### Final Synthesised Report

**Stocks Investment Overview and Optimal Timing**  

Stocks represent ownership in a company through shares, granting proportional rights to assets, earnings, and voting power. Shareholders may receive dividends and benefit from stock price appreciation, though ownership stakes depend on share counts and market dynamics.  

Determining the right time to invest involves analyzing market conditions (bull/bear phases), economic indicators (interest rates, GDP), company fundamentals (earnings, growth), valuation metrics (P/E ratios), and investor sentiment. While timing the market is inherently challenging, long-term goals, risk tolerance, and diversification are critical. Consistent dollar-cost averaging and disciplined strategies mitigate timing risks.  

Strategies to identify optimal entry points include fundamental analysis (evaluating financial health and macroeconomic trends), technical analysis (studying price patterns), and risk management techniques like diversification and stop-loss orders. Monitoring indicators such as PMIs, CPI, and Fed policies helps align investments with broader economic signals.  

Market conditions and economic indicators significantly influence stock decisions. Strong GDP growth, low unemployment, and corporate profitability often drive bullish sentiment, while inflation, rising rates, or recessions may prompt caution. However, these signals are not perfect predictors, underscoring the need for a balanced approach that combines adaptability, discipline, and alignment with personal financial objectives. Ultimately, long-term investing and strategic diversification are key to navigating market volatility.  

(Word count: 200)

---
## Exercise#5- Evaluator-Optimizer 
with custom rubric


In [18]:

WRITING_TASK    = "Explain how attention mechanisms work in transformer models, for a junior developer."
SCORE_THRESHOLD = 7
MAX_ITERATIONS  = 4

# ── Generator: produce or refine a draft ──────────────────────────────────────
def generate(task: str, feedback: str | None = None) -> str:
    messages = [
        {"role": "system", "content": "You are a technical educator. Write clearly and concisely (≤150 words)."},
        {"role": "user",   "content": task},
    ]
    if feedback:
        messages.append({"role": "user", "content": f"Improve your previous response based on this feedback:\n{feedback}"})
    return chat(messages)

# ── Evaluator: score on three axes ───────────────────────────────────────────
def evaluate(task: str, response: str) -> tuple[dict, str]:
    """
    Return ({'clarity': int, 'accuracy': int, 'brevity': int}, feedback string).
    """
    raw = chat(
        [
            {"role": "system", "content": (
                "You are a strict writing evaluator. "
                "Score the response from 1–10 on three axes: clarity, accuracy, brevity. "
                "Reply ONLY with JSON: {\"clarity\": <int>, \"accuracy\": <int>, \"brevity\": <int>, \"feedback\": \"<string>\"}"
            )},
            {"role": "user", "content": f"Task: {task}\n\nResponse:\n{response}"},
        ],
        temperature=0,
    ).strip()
    clean = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    parsed = json.loads(clean)
    scores = {axis: int(parsed[axis]) for axis in ["clarity", "accuracy", "brevity"]}
    return scores, parsed["feedback"]

# ── Optimization loop ─────────────────────────────────────────────────────────
feedback = None
for iteration in range(1, MAX_ITERATIONS + 1):
    draft = generate(WRITING_TASK, feedback)
    scores, feedback = evaluate(WRITING_TASK, draft)

    display(Markdown(
        f"### Iteration {iteration}\n"
        f"Scores → Clarity: {scores['clarity']}/10, Accuracy: {scores['accuracy']}/10, Brevity: {scores['brevity']}/10\n\n"
        f"**Draft:**\n\n{draft}\n\n"
        f"**Evaluator feedback:** {feedback}"
    ))

    # Check if all three scores meet threshold
    if all(score >= SCORE_THRESHOLD for score in scores.values()):
        display(Markdown(f"✅ **Threshold reached at iteration {iteration}. Final answer above.**"))
        break
else:
    display(Markdown(
        f"⚠️ Max iterations reached. Best scores → "
        f"Clarity: {scores['clarity']}, Accuracy: {scores['accuracy']}, Brevity: {scores['brevity']}"
    ))

### Iteration 1
Scores → Clarity: 8/10, Accuracy: 9/10, Brevity: 7/10

**Draft:**

Attention mechanisms in transformers let the model focus on relevant parts of input data. Imagine reading a sentence: instead of processing every word equally, the model identifies which words are most important for understanding the current word.  

Here’s how it works:  
1. **Query, Key, Value**: Each word is transformed into three vectors (query, key, value).  
2. **Dot Product**: The model calculates how well the current word’s query matches other words’ keys.  
3. **Weights**: These scores determine how much each word’s value contributes to the output.  
4. **Softmax**: Normalizes weights so they sum to 1, ensuring focus on key parts.  

This allows the model to dynamically prioritize information, improving tasks like translation or text generation. It’s like a spotlight that highlights relevant details in the input.

**Evaluator feedback:** Clear analogy and structured explanation, but could clarify the role of attention weights in combining values and mention scaling in dot product for completeness.

✅ **Threshold reached at iteration 1. Final answer above.**

---
## Exercise#6- Combined Patterns
Mini content pipeline that combines routing, orchestrator–workers, and evaluator–optimizer into one flow. 

In [6]:
from concurrent.futures import ThreadPoolExecutor
import json
import ollama
from IPython.display import display, Markdown

TOPIC = "How vector databases create business value for RAG applications"
SCORE_THRESHOLD = 8   # stop only when all three ≥ 8
MAX_ITERATIONS  = 4

CATEGORY = {
    "technical": "You are a senior technical writer. Be precise and structured.",
    "business": "You are a business strategist. Focus on value, ROI, and use cases."
}

OLLAMA_MODEL = "qwen3:8b"

def chat(messages, temperature=0):
    """Wrapper around Ollama chat API."""
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=messages,
        options={"temperature": temperature},
    )
    return response["message"]["content"]

# ── Routing ──────────────────────────────────────────────────────────
def classify_topic(topic: str) -> str:
    result = chat(
        [
            {"role": "system", "content": (
                "Classify the following topic as 'technical' or 'business'. "
                "Reply with the single word only: 'technical' or 'business'."
            )},
            {"role": "user", "content": topic},
        ],
        temperature=0).strip().lower()
    
    # Validate the reply
    if result in CATEGORY:
        return result
    # Fallback heuristic
    if any(word in topic.lower() for word in ["database", "rag", "ai", "system", "engineering"]):
        return "technical"
    return "business"

# ── Orchestrator-Workers ──────────────────────────────────────────
def generate_subtasks(topic: str) -> list[str]:
    raw = chat(
        [
            {"role": "system", "content": (
                "You are a research orchestrator. Give 3–4 focused sub-questions. "
                "Return ONLY valid JSON array of strings, no other text."
            )},
            {"role": "user", "content": topic},
        ],
        temperature=0,
    )
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        subtasks = json.loads(clean)
        if isinstance(subtasks, list) and all(isinstance(q, str) for q in subtasks):
            return subtasks
    except json.JSONDecodeError:
        pass
    # Fallback if parsing fails
    return [
        "What is a vector database?",
        "How do vector databases support RAG?",
        "What are the business benefits of using vector databases?"
    ]

def worker_answer(question: str) -> tuple[str, str]:
    answer = chat([
        {"role": "system", "content": "You are a concise expert. Answer in 3–5 sentences."},
        {"role": "user",   "content": question},
    ])
    return question, answer

def run_workers(subtasks: list[str]) -> dict:
    if not subtasks:
        raise ValueError("No subtasks generated")
    with ThreadPoolExecutor(max_workers=len(subtasks)) as pool:
        worker_results = dict(pool.map(lambda q: worker_answer(q), subtasks))
    return worker_results

# ── Orchestrator: synthesise into a final report ──────────────────────────────
def synthesize(topic: str, worker_text: str, category: str) -> str:
    final_report = CATEGORY[category]
    return chat([
        {"role": "system", "content": final_report},
        {"role": "user",   "content": f"Topic: {topic}\n\nResearch:\n{worker_text}\nWrite a clear 150-word explanation."},
    ])

# ── Evaluator-Optimizer ──────────────────────────────────────────
def evaluate(topic: str, response: str) -> tuple[dict, str]:
    raw = chat(
        [
            {"role": "system", "content": (
                "You are a strict writing evaluator. "
                "Score the response separately on clarity, accuracy, and brevity (each 1–10). "
                "Reply ONLY with JSON: "
                "{\"clarity\": <int>, \"accuracy\": <int>, \"brevity\": <int>, \"feedback\": \"text\"}"
            )},
            {"role": "user", "content": f"Topic: {topic}\n\nResponse:\n{response}"},
        ],
        temperature=0,
    ).strip()

    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        parsed = {"clarity": 5, "accuracy": 5, "brevity": 5, "feedback": "Evaluator returned invalid JSON"}
    scores = {
        "clarity": int(parsed.get("clarity", 0)),
        "accuracy": int(parsed.get("accuracy", 0)),
        "brevity": int(parsed.get("brevity", 0)),
    }
    return scores, parsed.get("feedback", "")

# ── Pipeline ──────────────────────────────────────────────────────────
category = classify_topic(TOPIC)
subtasks = generate_subtasks(TOPIC)
print("DEBUG subtasks:", subtasks)

worker_results = run_workers(subtasks)
worker_text = "\n\n".join(f"**Q: {q}**\n\n{a}" for q, a in worker_results.items())

draft = synthesize(TOPIC, worker_text, category)

feedback = None
for iteration in range(1, MAX_ITERATIONS + 1):
    if iteration > 1:
        draft = chat([
            {"role": "system", "content": f"{CATEGORY[category]} Revise carefully based on evaluator feedback."},
            {"role": "user", "content": f"Feedback:\n{feedback}\n\nPrevious draft:\n{draft}"}
        ])
    scores, feedback = evaluate(TOPIC, draft)

    display(Markdown(
        f"### Iteration {iteration}\n\n"
        f"**Scores:** {scores}\n\n"
        f"{draft}\n\n"
        f"**Evaluator feedback:** {feedback}"
    ))

    if all(s >= SCORE_THRESHOLD for s in scores.values()):
        display(Markdown(f"✅ **Final output is seen and the criteria are achieved.**"))
        break
else:
    display(Markdown(f"⚠️ Max iterations reached."))


DEBUG subtasks: ['How do vector databases enhance the efficiency of document retrieval in RAG systems, leading to faster and more accurate responses?', 'In what ways do vector databases support scalable data management, enabling RAG applications to handle growing data volumes without compromising performance?', 'What business advantages arise from improved query accuracy and relevance provided by vector databases in RAG applications?', 'How do vector databases contribute to cost reduction in maintaining and querying large-scale document datasets for RAG systems?']


### Iteration 1

**Scores:** {'clarity': 9, 'accuracy': 10, 'brevity': 8}

Vector databases unlock significant business value for RAG applications by enhancing retrieval efficiency, scalability, and cost-effectiveness. By converting documents into semantic vectors, they enable rapid, context-aware similarity searches, improving response accuracy and reducing latency. This speeds up RAG workflows, boosting user satisfaction and operational efficiency. Scalable indexing (e.g., IVF, HNSW) and distributed architectures allow systems to handle growing datasets without performance degradation, ensuring real-time responsiveness. Enhanced query accuracy drives trust and engagement, fostering customer loyalty and reducing misinformation risks. Additionally, vector databases lower costs through optimized storage, compression, and approximate nearest-neighbor algorithms, minimizing computational and infrastructure expenses. These advantages enable RAG systems to scale efficiently, deliver precise insights, and support data-driven decisions—all critical for maintaining competitive edge and maximizing ROI in AI-driven applications.

**Evaluator feedback:** The response is clear and accurate, effectively explaining how vector databases enhance RAG applications. It could be slightly more concise in the concluding sentence to improve brevity.

✅ **Final output is seen and the criteria are achieved.**